# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [2]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [3]:
df_housing = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
df_housing.head(1)

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000


In [4]:
df_housing["Bathrooms"] = df_housing["Full Bath"] + df_housing["Half Bath"] * 0.5

house0 = df_housing.loc[0]
show_vars = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Fireplaces", "Garage Type", "Bldg Type", "House Style", "Neighborhood", "Year Built", "SalePrice"]
house0[show_vars]

,0
Gr Liv Area,1656
Bedroom AbvGr,3
Bathrooms,1.0
Fireplaces,2
Garage Type,Attchd
Bldg Type,1Fam
House Style,1Story
Neighborhood,NAmes
Year Built,1960
SalePrice,215000


In [5]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X = df_housing[features].astype(float)

X_z = (X - X.mean()) / X.std()

diff = X_z - X_z.loc[0]
df_housing["euclid_z"] = np.sqrt((diff ** 2).sum(axis=1))
df_housing["manhattan"] = diff.abs().sum(axis=1)

In [6]:
df_cheaper_sales_price = df_housing[df_housing["SalePrice"]<house0["SalePrice"]]

df_cheaper_sales_price.sort_values("euclid_z")[show_vars + ["euclid_z"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,euclid_z
1226,1661,3,1.0,1,BuiltIn,1Fam,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1,Attchd,1Fam,1Story,NAmes,1953,153000,0.017804
291,1666,3,1.0,1,NaN,1Fam,1.5Fin,SWISU,1931,100000,0.019782
758,1666,3,1.0,0,Detchd,1Fam,1.5Fin,IDOTRR,1927,135000,0.019782
1357,1666,3,1.0,1,Detchd,1Fam,2Story,OldTown,1925,161000,0.019782


In [7]:
df_cheaper_sales_price = df_housing[df_housing["SalePrice"]<house0["SalePrice"]]

df_cheaper_sales_price.sort_values("manhattan")[show_vars + ["manhattan"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,manhattan
1226,1661,3,1.0,1,BuiltIn,1Fam,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1,Attchd,1Fam,1Story,NAmes,1953,153000,0.017804
1357,1666,3,1.0,1,Detchd,1Fam,2Story,OldTown,1925,161000,0.019782
758,1666,3,1.0,0,Detchd,1Fam,1.5Fin,IDOTRR,1927,135000,0.019782
291,1666,3,1.0,1,NaN,1Fam,1.5Fin,SWISU,1931,100000,0.019782


**The results are the same whether we use euclid or manhattan**

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [8]:
# Numerical features
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Fireplaces"]
Y = df_housing[features].astype(float)

# Standardize numerical features
Y_z = (Y - Y.mean()) / Y.std()

# Convert House Style to dummy variables
style = pd.get_dummies(
    df_housing["House Style"],
    prefix="style",
    dtype=float
)

# Add House Style variables to the standardized features
Y_z = pd.concat([Y_z, style], axis=1)

# Calculate distance from house 0
diff2 = Y_z - Y_z.loc[0]

df_housing["euclid_z_2"] = np.sqrt((diff2 ** 2).sum(axis=1))
df_housing["manhattan_2"] = diff2.abs().sum(axis=1)

In [9]:
df_cheaper_sales_price = df_housing[df_housing["SalePrice"]<house0["SalePrice"]]
# Find closest cheaper houses
df_cheaper_sales_price.sort_values("euclid_z_2")[
    show_vars + ["euclid_z_2"]
].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,euclid_z_2
618,1644,3,1.0,2,Attchd,1Fam,1Story,NAmes,1953,167000,0.023738
314,1687,3,1.0,2,Detchd,1Fam,1Story,Timber,1948,160000,0.061324
1013,1474,3,1.0,2,Attchd,1Fam,1Story,Gilbert,1952,115000,0.360033
1896,1429,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,181900,0.449052
1966,1377,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,156500,0.551919


In [10]:
# Find closest cheaper houses
df_cheaper_sales_price.sort_values("manhattan_2")[
    show_vars + ["manhattan_2"]
].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,manhattan_2
618,1644,3,1.0,2,Attchd,1Fam,1Story,NAmes,1953,167000,0.023738
314,1687,3,1.0,2,Detchd,1Fam,1Story,Timber,1948,160000,0.061324
1013,1474,3,1.0,2,Attchd,1Fam,1Story,Gilbert,1952,115000,0.360033
1896,1429,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,181900,0.449052
1966,1377,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,156500,0.551919


**The houses are continuing to get more similar to our dream home**

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [11]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Fireplaces"]
Z = df_housing[features].astype(float)

# Standardize numerical features
Z_z = (Z - Z.mean()) / Z.std()


garage = pd.get_dummies(
    df_housing["Garage Type"],
    prefix="Garage_Type",
    dtype=float
)

building = pd.get_dummies(
    df_housing["Bldg Type"],
    prefix="Bldg_Type",
    dtype=float
)

Z_z = pd.concat([Z_z, style, garage, building], axis=1)

# Calculate distance from house 0
diff3 = Z_z - Z_z.loc[0]

df_housing["euclid_z_3"] = np.sqrt((diff3 ** 2).sum(axis=1))
df_housing["manhattan_3"] = diff3.abs().sum(axis=1)


In [12]:
# Only look at houses cheaper than house 0
df_cheaper_sales_price = df_housing[
    df_housing["SalePrice"] < house0["SalePrice"]
]

# Find closest cheaper houses
df_cheaper_sales_price.sort_values("euclid_z_3")[
    show_vars + ["euclid_z_3"]
].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,euclid_z_3
618,1644,3,1.0,2,Attchd,1Fam,1Story,NAmes,1953,167000,0.023738
1013,1474,3,1.0,2,Attchd,1Fam,1Story,Gilbert,1952,115000,0.360033
1896,1429,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,181900,0.449052
1966,1377,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,156500,0.551919
145,1264,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,167500,0.775456


In [13]:
# Find closest cheaper houses
df_cheaper_sales_price.sort_values("manhattan_3")[
    show_vars + ["manhattan_3"]
].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Fireplaces,Garage Type,Bldg Type,House Style,Neighborhood,Year Built,SalePrice,manhattan_3
618,1644,3,1.0,2,Attchd,1Fam,1Story,NAmes,1953,167000,0.023738
1013,1474,3,1.0,2,Attchd,1Fam,1Story,Gilbert,1952,115000,0.360033
1896,1429,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,181900,0.449052
1966,1377,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,156500,0.551919
145,1264,3,1.0,2,Attchd,1Fam,1Story,NAmes,1960,167500,0.775456


**The quality of the matches are going up but the euclidean and manhattan scores are increasing because we are adding more variables.**

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [14]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [15]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

,California Polytechnic State University-San Luis Obispo
City,San Luis Obispo
State,CA
AdmissionRate,0.33
Undergraduates,21090.0
CarnegieClassification,Master's Colleges & Universities: Larger Programs
Ownership,Public
PCIP01,0.1084
PCIP03,0.0255
PCIP04,0.0441
PCIP05,0.0019


1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [47]:
vars=["AdmissionRate","Undergraduates"]
school = df_college[vars].astype(float)

# Standardize numerical features
school_z = (school - school.mean()) / school.std()

# Calculate distance from cal poly
diff_school = school_z - school_z.loc[school_name]

df_college["euclid"] = np.sqrt((diff_school ** 2).sum(axis=1))
df_college["manhattan"] = diff_school.abs().sum(axis=1)

In [48]:
df_college.sort_values("euclid")[
    vars + ["euclid", "manhattan"]
].head().reset_index()

,Institution,AdmissionRate,Undergraduates,euclid,manhattan
0,California Polytechnic State University-San Lu...,0.3300,21090.0,0.000000,0.000000
1,University of California-Santa Barbara,0.2918,23081.0,0.309162,0.429191
2,DeVry University-Illinois,0.4552,19729.0,0.593121,0.741854
3,University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846,0.746376
4,Clemson University,0.4922,21577.0,0.736788,0.796807


**The 4 schools above are the most similar to Cal Poly if only using Admission Rate and Undergraduate counts based on their standardized euclidean and manhattan distances values**

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [45]:
Carnegie = pd.get_dummies(
    df_college["CarnegieClassification"],
    prefix="Classification: ",
    dtype=float
)

vars=["AdmissionRate","Undergraduates"]
school = df_college[vars].astype(float)

# Standardize numerical features
school_z = (school - school.mean()) / school.std()

school_z2 = pd.concat([school_z, Carnegie], axis=1)

# Calculate distance from house 0
school_diff2 = school_z2 - school_z2.loc[school_name]

df_college["euclid2"] = np.sqrt((school_diff2 ** 2).sum(axis=1))
df_college["manhattan2"] = school_diff2.abs().sum(axis=1)

In [46]:
df_college.sort_values("euclid2")[
    vars + ["euclid2", "manhattan2"]
].head().reset_index()

,Institution,AdmissionRate,Undergraduates,euclid2,manhattan2
0,California Polytechnic State University-San Lu...,0.3300,21090.0,0.000000,0.000000
1,DeVry University-Illinois,0.4552,19729.0,0.593121,0.741854
2,CUNY Hunter College,0.4590,17293.0,0.761441,1.072635
3,CUNY Bernard M Baruch College,0.5056,15483.0,1.073601,1.516545
4,CUNY John Jay College of Criminal Justice,0.4458,12834.0,1.184989,1.586893


**the school most similar to Cal Poly is DeVry University-Illinois, this is based on euclidean distance and manhattan**

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [43]:
vars2=df_college.columns[6:44].tolist()
school = df_college[vars2].astype(float)

# Standardize numerical features
school_z3 = (school - school.mean()) / school.std()

# Calculate distance from cal poly
diff_school3 = school_z3 - school_z3.loc[school_name]

df_college["euclid3"] = np.sqrt((diff_school3 ** 2).sum(axis=1))
df_college["manhattan3"] = diff_school3.abs().sum(axis=1)

In [44]:
df_college.sort_values("euclid3")[
    vars2 + ["euclid3", "manhattan3"]
].head().reset_index()

,Institution,PCIP01,PCIP03,PCIP04,PCIP05,PCIP09,PCIP10,PCIP11,PCIP12,PCIP13,...,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54,euclid3,manhattan3
0,California Polytechnic State University-San Lu...,0.1084,0.0255,0.0441,0.0019,0.0353,0.0175,0.0326,0.0000,0.0130,...,0.0,0.0,0.0,0.0000,0.0130,0.0060,0.1637,0.0110,0.000000,0.000000
1,Iowa State University,0.1013,0.0131,0.0163,0.0009,0.0392,0.0000,0.0428,0.0000,0.0473,...,0.0,0.0,0.0,0.0000,0.0399,0.0131,0.1651,0.0069,1.986207,7.559127
2,California State Polytechnic University-Pomona,0.0384,0.0000,0.0324,0.0043,0.0306,0.0000,0.0415,0.0000,0.0209,...,0.0,0.0,0.0,0.0000,0.0295,0.0000,0.2937,0.0094,2.241726,7.710675
3,Texas A & M University-College Station,0.0847,0.0241,0.0111,0.0001,0.0397,0.0000,0.0334,0.0000,0.0272,...,0.0,0.0,0.0,0.0048,0.0010,0.0582,0.1593,0.0094,2.258942,8.063105
4,Mississippi State University,0.0619,0.0263,0.0208,0.0000,0.0360,0.0000,0.0201,0.0002,0.0712,...,0.0,0.0,0.0,0.0000,0.0102,0.0040,0.1825,0.0069,2.335288,9.695891


**The school closest now is Iwoa State University and this is determined through euclidean and manhattan**